# So sánh dataset đã tạo với paper DeepSMOTE

Notebook này **chỉ đọc/tạo dataset rồi so sánh** — không train autoencoder, không SMOTE, không GAN.
Dùng lại các hàm trong `src/visualization/report.py` và `src/preprocessing/create_imbalanced_dataset.py`, không viết lại logic.

Chạy được trên **local (VS Code/Jupyter), Google Colab, và Kaggle Notebook** — cell đầu tiên tự nhận diện môi trường, tự clone repo về và cài dependency nếu cần.

**Cách chạy:**
- Local: chọn kernel `.venv`, chạy từ trên xuống.
- Colab: `Runtime > Run all`.
- Kaggle: bật **Internet: On** trong Settings (góc phải) trước, rồi **Run All**. Không bật Internet thì bước `git clone` và tải dataset gốc (MNIST/CIFAR-10/...) sẽ lỗi.

Dataset nào chưa có trong `data/processed/` sẽ được tự tạo lại (tải dataset gốc + sample theo paper) ngay trong cell "Chọn dataset và seed", vì Colab/Kaggle đều xóa hết file sau mỗi phiên.

Sửa `REPO_URL` bên dưới nếu bạn fork repo sang tài khoản khác.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or "KAGGLE_URL_BASE" in os.environ
REPO_URL = "https://github.com/ColorOfDreams/GAN.git"

if IN_COLAB:
    REPO_DIR = Path("/content/GAN")
elif IN_KAGGLE:
    REPO_DIR = Path("/kaggle/working/GAN")
else:
    REPO_DIR = Path.cwd()

if IN_COLAB or IN_KAGGLE:
    if not (REPO_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
    os.chdir(REPO_DIR)
    # torch/torchvision đã có sẵn trên Colab/Kaggle và thỏa điều kiện >=, pip sẽ bỏ qua, không cài lại bản to
    subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)

SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Thư mục project:", REPO_DIR)
print("Colab:", IN_COLAB, "| Kaggle:", IN_KAGGLE)

## (Tùy chọn, chỉ Colab) Lưu dữ liệu qua Google Drive

Colab xóa hết file sau khi hết phiên, nên mỗi lần mở lại phải tải MNIST/CIFAR-10/... và sample lại từ đầu. Nếu muốn giữ `data/processed/` (và `reports/`) giữa các lần chạy, đặt `USE_DRIVE = True` ở cell dưới rồi chạy — notebook sẽ mount Drive và trỏ `data/processed`, `reports/` vào đó qua symlink. Bỏ qua bước này nếu chỉ chạy thử nhanh, hoặc nếu đang chạy trên Kaggle (Kaggle không có Drive — dùng "Kaggle Dataset" nếu muốn giữ dữ liệu giữa các lần chạy, xem README).

In [ ]:
USE_DRIVE = False

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

    drive_root = Path("/content/drive/MyDrive/GAN_deepsmote_data")
    for name in ("processed", "raw"):
        (drive_root / name).mkdir(parents=True, exist_ok=True)
        local_path = REPO_DIR / "data" / name
        local_path.parent.mkdir(parents=True, exist_ok=True)
        if local_path.is_symlink() or local_path.exists():
            continue
        os.symlink(drive_root / name, local_path, target_is_directory=True)
    print("Đã trỏ data/processed và data/raw vào Google Drive:", drive_root)

In [ ]:
from preprocessing.config import DATASET_CONFIGS
from preprocessing.create_imbalanced_dataset import create_imbalanced_dataset
from visualization.report import (
    load_processed_dataset,
    compare_with_deepsmote,
    comparison_all_match,
    format_comparison_table,
    summarize_dataset,
    format_overview_table,
    plot_class_distribution,
    plot_sample_grid,
)

## Chọn dataset và seed

`DATASET` là một trong: `mnist`, `fashion_mnist`, `cifar10`, `svhn`, `celeba`. Nếu dataset này chưa được tạo trong `data/processed/`, cell dưới sẽ tự chạy `create_imbalanced_dataset` (tải dataset gốc + sample theo đúng số liệu paper DeepSMOTE) trước khi load. CelebA có thể lỗi vì Google Drive giới hạn lượt tải — xem README nếu gặp lỗi này.

In [ ]:
DATASET = "mnist"
SEED = 42

try:
    ds = load_processed_dataset(DATASET, SEED)
except FileNotFoundError:
    print(f"Chưa có dataset '{DATASET}' (seed={SEED}), đang tạo...")
    create_imbalanced_dataset(DATASET, seed=SEED)
    ds = load_processed_dataset(DATASET, SEED)

print(f"Đã load {ds.cfg.display_name}: train={len(ds.train_y)} mẫu, test={len(ds.test_y)} mẫu")

## Bảng so sánh số mẫu mỗi class: paper DeepSMOTE vs. dataset đã tạo

In [ ]:
rows = compare_with_deepsmote(ds)
print(format_comparison_table(rows))
if not comparison_all_match(rows):
    print()
    print("Một số class lệch so với paper DeepSMOTE, xem bảng ở trên.")

## Biểu đồ phân bố class (paper vs. thực tế)

In [ ]:
fig = plot_class_distribution(ds, rows)

## Lưới ảnh mẫu theo từng class

In [ ]:
fig = plot_sample_grid(ds, samples_per_class=6, seed=SEED)

## So sánh nhanh các dataset đã tạo

Chỉ kiểm tra dataset **đã có sẵn** trong `data/processed/` (không tự tạo hết, vì CIFAR-10/SVHN/CelebA tải khá lâu). Muốn thêm dataset nào vào bảng này, chạy cell "Chọn dataset và seed" ở trên với `DATASET` tương ứng trước.

In [ ]:
summaries = {key: summarize_dataset(key, SEED) for key in DATASET_CONFIGS}
print(format_overview_table(summaries))